# Relevé Vectorizer

This notebook transforms raw relevé data (species occurrences per plot) into numerical feature vectors suitable for machine learning classification.

## 1. Load Dependencies

In [ ]:
import pandas as pd
import numpy as np
import json

## 2. Load Raw Relevé Data

In [ ]:
df_pirineus = pd.read_csv('matriu_dades/ReleveSenseDAdes2.txt', sep=',', header=None,
                         names=['Id', 'Sp_id', 'Index', 'Releve_id'])
df_pirineus.head()

## 3. Load Species Index Dictionary

The species index dictionary maps each species ID to a fixed position in the feature vector.

In [ ]:
with open('matriu_dades/sp_index_dic.txt', 'r') as file:
    sp_index_dic = json.load(file)

print(f"Total species in index: {len(sp_index_dic)}")

## 4. Filter Out Unknown Species

Remove species not present in the reference species index.

In [ ]:
values_to_remove = [sp for sp in df_pirineus['Sp_id'] if sp not in sp_index_dic.keys()]
df_filtered = df_pirineus[~df_pirineus['Sp_id'].isin(values_to_remove)]

print(f"Removed {len(set(values_to_remove))} unknown species")
print(f"Remaining records: {len(df_filtered)}")

## 5. Build Relevé-to-ID Mapping

In [ ]:
id_releve_dic = dict(zip(df_filtered['Id'], df_filtered['Releve_id']))

## 6. Identify Record Ranges per Relevé

Find the start and end indices for each relevé's species records in the dataframe.

In [ ]:
def find_repetition_indices(lst):
    """Find start and end indices for consecutive groups of the same element."""
    repetition_indices = {}
    current_start = 0
    current_element = None

    for i, element in enumerate(lst):
        if element != current_element:
            if current_element is not None:
                repetition_indices[current_element] = (current_start, i - 1)
            current_start = i
            current_element = element

    if current_element is not None:
        repetition_indices[current_element] = (current_start, len(lst) - 1)

    return repetition_indices

indices_dict = find_repetition_indices(list(df_filtered['Id']))
print(f"Total relev?s to vectorize: {len(indices_dict)}")

## 7. Vectorize Relevés

Convert each relevé into a sparse vector where each position represents a species and the value represents its abundance index.

In [ ]:
sp_real_length = len(sp_index_dic)
vector_dic = {}

for key, value in indices_dict.items():
    my_vector = np.zeros(sp_real_length)
    start, end = value
    
    sp_id_list = list(df_filtered['Sp_id'])[start:end + 1]
    indices_to_set = [sp_index_dic[sp] for sp in sp_id_list]
    values_to_set = list(df_filtered['Index'])[start:end + 1]
    
    my_vector[indices_to_set] = values_to_set
    vector_dic[key] = [my_vector, id_releve_dic[key]]

print(f"Vectorized {len(vector_dic)} relev?s with {sp_real_length} features each")

## 8. Save Vectorized Data

In [ ]:
# Convert numpy arrays to lists for JSON serialization
for key, value in vector_dic.items():
    if isinstance(value[0], np.ndarray):
        vector_dic[key][0] = value[0].tolist()

file_path = 'matriu_dades/releves_sense_alianca_vectors.txt'
with open(file_path, 'w') as file:
    json.dump(vector_dic, file, indent=2)

print(f"Saved vectorized relev?s to {file_path}")

## 9. Verify Output

In [ ]:
# Reload and verify
with open(file_path, 'r') as file:
    loaded_vector_dic = json.load(file)

df = pd.DataFrame.from_dict(loaded_vector_dic, orient='index', columns=['List', 'Label'])
df_vectors = pd.DataFrame(df['List'].tolist(), index=df.index)
df_vectors.columns = [f'Column_{i+1}' for i in range(len(df_vectors.columns))]

labels = [loaded_vector_dic[idx][1] for idx in df_vectors.index]
df_vectors['Target'] = labels

print(f"Output shape: {df_vectors.shape}")
df_vectors.head()